# PMF and CDF of a discrete random variable
Rolling dice by simulation, estimating the PMF from counts, and turning it into a CDF.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

# fixed seed: the same 'random' rolls on every run
rng = np.random.default_rng(42)

## 1. One die, rolled 10,000 times

In [ ]:
# 10,000 rolls; integers(1, 7) gives 1 to 6 (the upper end is excluded)
rolls = rng.integers(1, 7, size=10_000)
print(rolls[:5])

In [ ]:
# count each face and divide by the number of rolls: estimated probabilities
pmf_one = pd.Series(rolls).value_counts(normalize=True).sort_index()
print(pmf_one.round(4))
print('exact 1/6 =', round(1 / 6, 4))

In [ ]:
px.bar(x=pmf_one.index, y=pmf_one.values, labels={'x': 'face', 'y': 'probability'},
       title='Estimated PMF of one die')

In [ ]:
# law of large numbers: 1,000 repeats of only 100 rolls each
r = np.random.default_rng(0)
shares = np.array([pd.Series(r.integers(1, 7, 100)).value_counts(normalize=True)
                   .reindex(range(1, 7), fill_value=0) for _ in range(1000)])
print('median lowest face share :', np.median(shares.min(axis=1)))
print('median highest face share:', np.median(shares.max(axis=1)))
print('10,000 rolls, largest gap from 1/6:', round(abs(pmf_one - 1 / 6).max(), 4))

## 2. The sum of two dice

In [ ]:
# roll two dice 10,000 times and add the faces
sums = rng.integers(1, 7, size=10_000) + rng.integers(1, 7, size=10_000)
print(sums[:5])
pmf_two = pd.Series(sums).value_counts(normalize=True).sort_index()
print(pmf_two.round(4))

In [ ]:
# the exact PMF: (6 - |x - 7|) / 36 for x = 2, ..., 12
x = np.arange(2, 13)
exact = (6 - np.abs(x - 7)) / 36
print(pd.DataFrame({'simulated': pmf_two.values, 'exact': exact}, index=x).round(4))
# the two conditions of a PMF
print('all >= 0:', (exact >= 0).all(), ' sum:', exact.sum())

## 3. Bernoulli and binomial PMFs

In [ ]:
print(stats.bernoulli(0.3).pmf([0, 1]))          # P(X=0), P(X=1)
print(stats.binom(4, 0.5).pmf([0, 1, 2, 3, 4]))   # 4 coin tosses, number of heads

## 4. The CDF: a running total of the PMF

In [ ]:
cdf_two = pmf_two.cumsum()
print(cdf_two.round(3))
# exact values
exact_cdf = np.cumsum(exact)
print('P(X <= 9) =', exact_cdf[x == 9][0], '=', 30 / 36)
print('P(X <= 5) =', exact_cdf[x == 5][0], '=', 10 / 36)
print('P(5 < X <= 9) =', 30 / 36 - 10 / 36)

In [ ]:
# PMF and CDF one above the other; the CDF is drawn as steps
fig = go.Figure()
fig.add_bar(x=x, y=exact, name='PMF  P(X = x)')
fig.add_scatter(x=np.r_[1, x, 13], y=np.r_[0, exact_cdf, 1], mode='lines', line_shape='hv',
                name='CDF  P(X <= x)')
fig.update_layout(xaxis_title='sum of two dice', yaxis_title='probability')
fig

In [ ]:
# the CDF is defined for every x, not only the integers: F(7.5) = F(7)
F = stats.rv_discrete(values=(x, exact)).cdf
print(F(7), F(7.5), F(1.9), F(12))